# 13. Peak Overlap Validation: SCENIC+ Inputs vs Cross-Species Consensus

**Goal:** Validate that SCENIC+ input peaks (DARs + Topics) are accounted for (≥90%) in our cross-species consensus peak sets (v3). Build a mapping table from SCENIC+ regions to consensus peak IDs.

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import pyranges as pr
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## Configuration

In [2]:
# --- Paths (edit these) ---
BASE_DIR = "/home/jjanssens/jjans/analysis/adult_intestine"

# Cross-species consensus peaks (v3)
CONSENSUS_DIR = os.path.join(BASE_DIR, "peaks/cross_species_consensus_v3/10_final")

# SCENIC+ input region sets
REGION_SET_DIR = os.path.join(BASE_DIR, "atac")

# Species list
SPECIES_LIST = ["Human", "Gorilla", "Chimpanzee", "Bonobo", "Macaque", "Marmoset"]

# Region set subdirectories to check
REGION_SET_TYPES = ["DARs_cell_type", "Topics_otsu"]

# Minimum overlap fraction to count as "covered" (1bp overlap)
MIN_OVERLAP_BP = 1

# Output directory for mapping files
OUTPUT_DIR = os.path.join(BASE_DIR, "peaks/cross_species_consensus_v3/scenic_mapping")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Consensus dir: {CONSENSUS_DIR}")
print(f"Region sets dir: {REGION_SET_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

Consensus dir: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/10_final
Region sets dir: /home/jjanssens/jjans/analysis/adult_intestine/atac
Output dir: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/scenic_mapping


## Helper functions

In [3]:
def load_bed3(path):
    """Load a 3-column BED file as PyRanges."""
    df = pd.read_csv(path, sep="\t", header=None, usecols=[0, 1, 2],
                     names=["Chromosome", "Start", "End"])
    return pr.PyRanges(df)


def load_bed4(path):
    """Load a 4-column BED file (with peak ID) as PyRanges."""
    df = pd.read_csv(path, sep="\t", header=None, usecols=[0, 1, 2, 3],
                     names=["Chromosome", "Start", "End", "peak_id"])
    return pr.PyRanges(df)


def overlap_fraction(query_pr, subject_pr):
    """Fraction of query intervals that overlap at least 1bp with subject."""
    n_query = len(query_pr)
    if n_query == 0:
        return 0.0, 0, 0
    overlapping = query_pr.count_overlaps(subject_pr)
    n_hit = int((overlapping.NumberOverlaps > 0).sum())
    return n_hit / n_query, n_hit, n_query


def map_to_consensus(query_pr, consensus_pr):
    """Join query regions with consensus peaks, returning mapping DataFrame."""
    # Add a query_region column for tracking
    qdf = query_pr.df.copy()
    qdf["query_region"] = qdf["Chromosome"] + ":" + qdf["Start"].astype(str) + "-" + qdf["End"].astype(str)
    query_with_id = pr.PyRanges(qdf)
    
    joined = query_with_id.join(consensus_pr, how="left")
    jdf = joined.df
    
    if len(jdf) == 0:
        return pd.DataFrame(columns=["query_region", "peak_id"])
    
    # Keep relevant columns — pyranges 0.0.x uses -1 for unmatched left joins
    result = jdf[["query_region", "peak_id"]].copy()
    result = result[~result["peak_id"].isin([-1, "-1"])].drop_duplicates()
    return result

## Load consensus peaks for all species

In [4]:
consensus_peaks = {}
for species in SPECIES_LIST:
    path = os.path.join(CONSENSUS_DIR, f"all_peaks_{species}.bed")
    consensus_peaks[species] = load_bed4(path)
    print(f"{species}: {len(consensus_peaks[species]):,} consensus peaks")

Human: 1,039,336 consensus peaks
Gorilla: 1,013,198 consensus peaks
Chimpanzee: 1,029,053 consensus peaks
Bonobo: 992,889 consensus peaks
Macaque: 982,905 consensus peaks
Marmoset: 971,722 consensus peaks


## Per-species overlap analysis

In [5]:
# Collect results
summary_rows = []
detail_rows = []  # per file

for species in SPECIES_LIST:
    cons = consensus_peaks[species]
    
    for rset_type in REGION_SET_TYPES:
        rset_dir = os.path.join(REGION_SET_DIR, f"region_sets_{species}", rset_type)
        if not os.path.isdir(rset_dir):
            print(f"  WARNING: {rset_dir} not found, skipping.")
            continue
        
        bed_files = sorted(glob.glob(os.path.join(rset_dir, "*.bed")))
        
        total_regions = 0
        total_hits = 0
        
        for bf in bed_files:
            name = Path(bf).stem
            query = load_bed3(bf)
            frac, n_hit, n_query = overlap_fraction(query, cons)
            total_regions += n_query
            total_hits += n_hit
            
            detail_rows.append({
                "species": species,
                "set_type": rset_type,
                "name": name,
                "n_regions": n_query,
                "n_overlapping": n_hit,
                "overlap_frac": frac,
            })
        
        agg_frac = total_hits / total_regions if total_regions > 0 else 0.0
        summary_rows.append({
            "species": species,
            "set_type": rset_type,
            "n_files": len(bed_files),
            "n_regions_total": total_regions,
            "n_overlapping_total": total_hits,
            "overlap_frac": agg_frac,
        })
        print(f"{species} / {rset_type}: {agg_frac:.1%} ({total_hits:,}/{total_regions:,}) across {len(bed_files)} files")

summary_df = pd.DataFrame(summary_rows)
detail_df = pd.DataFrame(detail_rows)
print("\nDone.")

Human / DARs_cell_type: 99.9% (410,717/411,315) across 32 files
Human / Topics_otsu: 89.6% (3,247,163/3,625,774) across 100 files
Gorilla / DARs_cell_type: 0.0% (0/606,715) across 31 files


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

## Summary table

In [ ]:
# Pivot: species × set_type
pivot = summary_df.pivot(index="species", columns="set_type", values="overlap_frac")
pivot = pivot.reindex(SPECIES_LIST)
pivot_styled = pivot.style.format("{:.1%}").background_gradient(cmap="RdYlGn", vmin=0.8, vmax=1.0)
display(pivot_styled)

# Also show raw counts
pivot_counts = summary_df.pivot(index="species", columns="set_type", values=["n_regions_total", "n_overlapping_total"])
display(pivot_counts)

## Barplot: overlap % per species

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

plot_data = summary_df.copy()
plot_data["overlap_pct"] = plot_data["overlap_frac"] * 100

sns.barplot(data=plot_data, x="species", y="overlap_pct", hue="set_type",
            order=SPECIES_LIST, ax=ax, palette="Set2")
ax.axhline(90, ls="--", color="red", alpha=0.7, label="90% threshold")
ax.set_ylabel("Overlap with consensus (%)")
ax.set_xlabel("")
ax.set_ylim(0, 105)
ax.legend(title="Region set")
ax.set_title("SCENIC+ input peaks covered by cross-species consensus (v3)")
plt.tight_layout()
plt.show()

## Heatmap: per cell-type DAR overlap by species

In [ ]:
# Filter to DARs only
dar_detail = detail_df[detail_df["set_type"] == "DARs_cell_type"].copy()

# Pivot: cell type × species
heatmap_data = dar_detail.pivot(index="name", columns="species", values="overlap_frac")
heatmap_data = heatmap_data.reindex(columns=SPECIES_LIST)
heatmap_data = heatmap_data.sort_index()

fig, ax = plt.subplots(figsize=(10, max(8, len(heatmap_data) * 0.35)))
sns.heatmap(heatmap_data, annot=True, fmt=".0%", cmap="RdYlGn",
            vmin=0.7, vmax=1.0, linewidths=0.5, ax=ax,
            cbar_kws={"label": "Overlap fraction"})
ax.set_title("DAR coverage by consensus peaks (per cell type × species)")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## Heatmap: per topic overlap by species

In [ ]:
topic_detail = detail_df[detail_df["set_type"] == "Topics_otsu"].copy()

heatmap_topics = topic_detail.pivot(index="name", columns="species", values="overlap_frac")
heatmap_topics = heatmap_topics.reindex(columns=SPECIES_LIST)

# Sort topics numerically
import re
def topic_sort_key(name):
    m = re.search(r"(\d+)", name)
    return int(m.group(1)) if m else 0

heatmap_topics = heatmap_topics.loc[sorted(heatmap_topics.index, key=topic_sort_key)]

fig, ax = plt.subplots(figsize=(10, max(8, len(heatmap_topics) * 0.22)))
sns.heatmap(heatmap_topics, cmap="RdYlGn", vmin=0.7, vmax=1.0,
            linewidths=0.3, ax=ax, cbar_kws={"label": "Overlap fraction"})
ax.set_title("Topic coverage by consensus peaks (per topic × species)")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## Flag categories below 90% coverage

In [ ]:
low_coverage = detail_df[detail_df["overlap_frac"] < 0.90].sort_values("overlap_frac")

if len(low_coverage) == 0:
    print("All region sets have >= 90% overlap with consensus peaks.")
else:
    print(f"{len(low_coverage)} region sets below 90% coverage:\n")
    display(low_coverage[["species", "set_type", "name", "n_regions", "n_overlapping", "overlap_frac"]]
            .reset_index(drop=True)
            .style.format({"overlap_frac": "{:.1%}"}))

## Build mapping: SCENIC+ region → consensus peak ID

For each species, map every SCENIC+ input region (DAR or Topic) to the overlapping consensus peak ID(s). This produces a lookup table used downstream to annotate eRegulons with peak IDs.

In [ ]:
for species in SPECIES_LIST:
    print(f"\n--- {species} ---")
    cons = consensus_peaks[species]
    
    all_query_dfs = []
    
    for rset_type in REGION_SET_TYPES:
        rset_dir = os.path.join(REGION_SET_DIR, f"region_sets_{species}", rset_type)
        if not os.path.isdir(rset_dir):
            continue
        for bf in sorted(glob.glob(os.path.join(rset_dir, "*.bed"))):
            name = Path(bf).stem
            qdf = pd.read_csv(bf, sep="\t", header=None, usecols=[0, 1, 2],
                              names=["Chromosome", "Start", "End"])
            qdf["source_file"] = rset_type
            qdf["source_name"] = name
            all_query_dfs.append(qdf)
    
    if not all_query_dfs:
        continue
    
    all_query = pd.concat(all_query_dfs, ignore_index=True)
    # Deduplicate identical regions (same region can appear in multiple DARs/Topics)
    all_unique = all_query[["Chromosome", "Start", "End"]].drop_duplicates()
    all_unique["query_region"] = (
        all_unique["Chromosome"] + ":" + all_unique["Start"].astype(str) + "-" + all_unique["End"].astype(str)
    )
    
    query_pr = pr.PyRanges(all_unique)
    mapping = map_to_consensus(query_pr, cons)
    
    # Add peak type classification
    if len(mapping) > 0:
        mapping["peak_type"] = mapping["peak_id"].apply(
            lambda x: "unified" if str(x).startswith("unified_") else
                      "human_specific" if str(x).startswith("human_peak_") else
                      "species_specific"
        )
    
    out_path = os.path.join(OUTPUT_DIR, f"scenic_to_consensus_mapping_{species}.tsv")
    mapping.to_csv(out_path, sep="\t", index=False)
    
    n_unique_regions = len(all_unique)
    n_mapped = mapping["query_region"].nunique() if len(mapping) > 0 else 0
    print(f"  Unique SCENIC+ regions: {n_unique_regions:,}")
    print(f"  Mapped to consensus:    {n_mapped:,} ({n_mapped/n_unique_regions:.1%})")
    print(f"  Saved: {out_path}")
    if len(mapping) > 0:
        print(f"  Peak type breakdown: {mapping['peak_type'].value_counts().to_dict()}")

## Summary

In [ ]:
print("=" * 60)
print("Peak Overlap Validation Summary")
print("=" * 60)
for _, row in summary_df.iterrows():
    status = "PASS" if row["overlap_frac"] >= 0.90 else "FAIL"
    print(f"  [{status}] {row['species']:12s} / {row['set_type']:20s}: "
          f"{row['overlap_frac']:.1%} ({row['n_overlapping_total']:,}/{row['n_regions_total']:,})")
print("\nMapping files saved to:", OUTPUT_DIR)